# Flows · **12 — The chart**

Monthly flows on and off the waiting list, April 2025 to the reporting month,
headline basis (with NHS England's estimates for missing trusts):

* **clock starts** above the zero line;
* **reported completed pathways** and **derived unreported removals** below it, as separate bands;
* the **net change in the list** in its own panel underneath, on its own scale.

Why two panels: net change runs to about ±110,000 against flows of about two
million, so on one axis the line would be flat. A second y-axis on the same
plot would let the reader compare two scales that aren't comparable. Two
panels sharing the month axis keep both honest.

The words are the findings' words. The chart says **reported** against
**unreported**, never "treated" against "bookkeeping": that claim failed in
notebook 11 (row 16). The title and the March note only print if their rows
in 11 say "holds".

Style: the same tokens and categorical slots as 09 (blue, orange, aqua; slots
1-3, validated as a set in the reference palette). The net change line wears
ink, not a series colour, because it's a different kind of mark. Light mode
only, for Medium.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import FIRST_MONTH, GOLD_DIR, REPORT_MONTH, mon_year, month_year  # noqa: E402

OUT = GOLD_DIR / "12_flows_chart"
OUT.mkdir(parents=True, exist_ok=True)
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#9a9891", "#e6e4de"
C_START, C_DONE, C_UNREP = "#2a78d6", "#eb6834", "#1baf7a"      # slots 1, 2, 3
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.unicode_minus": True})

def signed(x, fmt):
    return format(x, "+" + fmt).replace("-", "−")

## Step 0 — Gates, and the findings the words depend on

In [2]:
for g in [GOLD_DIR / "00_reconciliation" / "reconciliation.csv", GOLD_DIR / "10_flows" / "flows_gate.csv"]:
    assert pd.read_csv(g)["pass"].all(), f"{g.name}: a gate is failing"
fd = pd.read_csv(GOLD_DIR / "11_flows_findings" / "findings.csv").set_index("id")
holds = lambda i: fd.loc[i, "verdict"] == "holds"
assert holds(2), "title claim (about one exit in seven) no longer holds: reword the title"
assert holds(4) and holds(6), "March note (highest level and share) no longer holds: reword the note"
assert fd.loc[16, "verdict"] == "does not hold", "row 16 changed: re-check the chart's labels"

f = pd.read_csv(GOLD_DIR / "10_flows" / "flows_national.csv").query("basis == 'headline'").set_index("period")
f.index = pd.PeriodIndex(f.index, freq="M").to_timestamp()
share = 100 * f.unreported_published / (f.completed + f.unreported_published)
pooled = 100 * f.unreported_published.sum() / (f.completed.sum() + f.unreported_published.sum())
MAR = pd.Timestamp("2026-03-01")
f[["starts", "completed", "unreported_published", "net_change"]].round(0).tail(4)

,starts,completed,unreported_published,net_change
period,,,,
2026-06-01,1968600.0,1656538.0,318155.0,-6093.0
2026-07-01,1985434.0,1651478.0,277994.0,55961.0
2026-08-01,1717335.0,1430808.0,244679.0,41848.0
2026-09-01,1981844.0,1673669.0,302409.0,5766.0


## Step 1 — The chart

In [3]:
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 7.6), dpi=200, sharex=True,
                              gridspec_kw={"height_ratios": [3.2, 1.25], "hspace": 0.08})
fig.patch.set_facecolor(SURFACE)
x = f.index
W = 20                                      # bar width in days: leaves air between months
M = 1e6

for a in (ax, ax2):
    a.set_facecolor(SURFACE)
    a.axvspan(MAR - pd.Timedelta(days=15), MAR + pd.Timedelta(days=15), color=MUTED, alpha=0.13, lw=0, zorder=0)
    a.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    for s in ["top", "right", "left"]:
        a.spines[s].set_visible(False)
    a.spines["bottom"].set_color(GRID)
    a.tick_params(axis="y", labelsize=9, colors=INK2, length=0)

# Above zero: clock starts. Below zero: reported completions, then unreported
# removals stacked beneath them, with a surface-coloured edge as the gap.
ax.bar(x, f.starts / M, width=W, color=C_START, edgecolor=SURFACE, lw=1, zorder=2, label="Clock starts")
ax.bar(x, -f.completed / M, width=W, color=C_DONE, edgecolor=SURFACE, lw=1, zorder=2, label="Reported completed pathways")
ax.bar(x, -f.unreported_published / M, bottom=-f.completed / M, width=W, color=C_UNREP, edgecolor=SURFACE, lw=1, zorder=2,
       label="Derived unreported removals")
ax.axhline(0, color=INK2, lw=0.9, zorder=3)
ax.set_ylim(-2.45, 2.35)
ax.set_yticks([-2, -1, 0, 1, 2])
ax.set_yticklabels(["2.0m", "1.0m", "0", "1.0m", "2.0m"])
ax.text(x[0] - pd.Timedelta(days=14), 2.24, "On to the list", fontsize=8.5, color=INK2, va="center")
ax.text(x[0] - pd.Timedelta(days=14), -2.36, "Off the list", fontsize=8.5, color=INK2, va="center")

# One selective note: March, the month the findings single out.
mu = f.unreported_published[MAR]
ax.annotate(f"March 2026: {mu / 1e3:,.0f}k unreported,\n{share[MAR]:.1f}% of exits (highest in the window)",
            xy=(MAR, -(f.completed[MAR] + mu) / M), xytext=(pd.Timestamp("2025-11-05"), -2.22),
            fontsize=8.5, color=INK2, va="center", ha="right",
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))

# The legend sits above the plot, so it never covers a bar or the 'On to the list' label.
leg = fig.legend(*ax.get_legend_handles_labels(), loc="upper left", bbox_to_anchor=(0.075, 0.885), ncol=3, frameon=False,
                 fontsize=9, handlelength=1.2, columnspacing=1.6)
for t in leg.get_texts():
    t.set_color(INK)

# Lower panel: the net change in the list.
ax2.axhline(0, color=INK2, lw=0.9, zorder=2)
ax2.plot(x, f.net_change / 1e3, color=INK, lw=2, solid_capstyle="round", solid_joinstyle="round", zorder=3)
ax2.scatter(x, f.net_change / 1e3, s=16, color=INK, zorder=4)
ax2.set_ylim(-140, 140)
ax2.set_yticks([-100, 0, 100])
ax2.set_yticklabels([signed(v, ",.0f") + "k" if v else "0" for v in [-100, 0, 100]])
ax2.text(x[0] - pd.Timedelta(days=14), 128, "Net change in the list", fontsize=8.5, color=INK2, va="center")
ax2.annotate(f"{signed(f.net_change[MAR] / 1e3, ',.0f')}k", xy=(MAR, f.net_change[MAR] / 1e3),
             xytext=(MAR + pd.Timedelta(days=22), f.net_change[MAR] / 1e3), fontsize=8.5, color=INK2, va="center")

ticks = pd.date_range(f"{FIRST_MONTH}-01", f"{REPORT_MONTH}-01", freq="2MS")
ax2.set_xticks(ticks)
ax2.set_xticklabels([d.strftime("%b\n%Y") if (d == ticks[0] or d.month == 1) else d.strftime("%b") for d in ticks],
                    fontsize=9, color=INK2)
ax2.tick_params(axis="x", length=0)
ax.tick_params(axis="x", length=0)
ax2.set_xlim(x[0] - pd.Timedelta(days=20), x[-1] + pd.Timedelta(days=20))

# 'About one in seven' is the pooled and median share (findings row 2). Single months
# range from about one in eight to one in six, so the title doesn't say 'every month'.
fig.text(0.02, 0.975, "About one exit from the waiting list in seven goes unreported",
         fontsize=14.5, color=INK, ha="left", va="top", weight="bold")
fig.text(0.02, 0.937, f"RTT waiting list, England, {month_year(FIRST_MONTH)} to {month_year(REPORT_MONTH)}: {pooled:.1f}% of exits "
         f"were not reported as completed pathways\n({share.min():.1f}% to {share.max():.1f}% in single months). "
         f"March 2026 (shaded) was the highest month.",
         fontsize=9.5, color=INK2, ha="left", va="top")
fig.text(0.02, 0.072,
         "Derived unreported removals are NHS England's balancing item: list at start + clock starts − reported completions "
         "− list at end.\nThey are not the same as validation removals. Reported completions are not all treatment either: "
         "a clock also stops on a decision\nnot to treat, a declined offer, active monitoring or a patient's death.",
         fontsize=7.6, color=INK2, ha="left", va="bottom")
fig.text(0.02, 0.012, f"Headline basis, including NHS England's estimates for trusts that did not submit.\nSource: NHS England "
         f"RTT Overview time series ({mon_year(REPORT_MONTH)} release); identity rebuilt and checked month by month.",
         fontsize=7.6, color=INK2, ha="left", va="bottom")
fig.subplots_adjust(left=0.08, right=0.98, top=0.845, bottom=0.185)
fig.savefig(OUT / "flows_monthly.png", dpi=200, facecolor=SURFACE)
plt.close(fig)
print("wrote", (OUT / "flows_monthly.png").name)

wrote flows_monthly.png


## Step 2 — Alt text, from the same numbers

In [4]:
lo, hi = f.unreported_published.idxmin(), f.unreported_published.idxmax()
alt = (f"Bar chart in two panels, {month_year(FIRST_MONTH)} to {month_year(REPORT_MONTH)}. Top panel: each month's clock starts "
       f"above zero (between {f.starts.min() / M:.2f} and {f.starts.max() / M:.2f} million) and, below zero, reported completed "
       f"pathways ({f.completed.min() / M:.2f} to {f.completed.max() / M:.2f} million) with derived unreported removals beneath them "
       f"({f.unreported_published.min() / 1e3:,.0f} thousand in {lo.strftime('%B %Y')} to "
       f"{f.unreported_published.max() / 1e3:,.0f} thousand in {hi.strftime('%B %Y')}). Across the period {pooled:.1f}% of exits "
       f"were unreported; March 2026 was the highest at {share[MAR]:.1f}%. Bottom panel: the net change in the list, between "
       f"{f.net_change.min() / 1e3:,.0f} thousand and {signed(f.net_change.max() / 1e3, ',.0f')} thousand a month, falling "
       f"{-f.net_change[MAR] / 1e3:,.0f} thousand in March 2026.")
(OUT / "alt_text.md").write_text(f"# Alt text\n\n## flows_monthly.png\n\n{alt}\n")
print(alt)

Bar chart in two panels, April 2025 to September 2026. Top panel: each month's clock starts above zero (between 1.63 and 1.99 million) and, below zero, reported completed pathways (1.40 to 1.71 million) with derived unreported removals beneath them (209 thousand in August 2025 to 352 thousand in March 2026). Across the period 14.6% of exits were unreported; March 2026 was the highest at 17.1%. Bottom panel: the net change in the list, between -110 thousand and +112 thousand a month, falling 110 thousand in March 2026.
